# 🧠 Prompt Quality Scoring Agent

**Built with LangChain | Runs on Google Colab**

Evaluates any AI prompt across **5 quality criteria**, returns a **score out of 10**, explains weaknesses, and gives **actionable improvement suggestions**.

---

## 📋 Table of Contents
1. [Installation](#section-1)
2. [Configuration & API Key Setup](#section-2)
3. [Agent Core — Data Models & Chain](#section-3)
4. [Display & Helper Functions](#section-4)
5. [Interactive Demo](#section-5)
6. [Batch Test Suite (10 Prompts)](#section-6)

---

### 📊 Evaluation Criteria

| Criterion | What's Evaluated |
|---|---|
| **Clarity** | Clear goal, unambiguous intent |
| **Specificity / Details** | Sufficient detail and requirements |
| **Context** | Background, audience, use case |
| **Output Format & Constraints** | Format, tone, length expectations |
| **Persona Defined** | Role assigned to the AI |

> **Final Score** = Average of all 5 criterion scores

---
## ⚙️ Section 1 — Installation

Run this cell once. It installs all required packages.

In [24]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
# Uses subprocess.check_call instead of !pip backslash continuations.
# Backslash-continued !pip lines in Colab can pass empty strings
# as package names, causing: 'Invalid requirement: \"\"'
#
# After this cell finishes:
#   Runtime → Restart runtime  (then run from Cell 2 onward)

import sys, subprocess

PACKAGES = [
    "langchain>=0.2.0",
    "langchain-google-genai>=2.0.0",  # Gemini via LangChain
    "langchain-core>=0.2.0",
    "google-genai>=1.0.0",     # Google GenAI SDK
    "pydantic>=2.0.0",
    "rich>=13.0.0",
    "tenacity>=8.2.0",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + PACKAGES
)

print("\u2705 All packages installed successfully.")
print("\n\u26a0\ufe0f  Next step: Runtime \u2192 Restart runtime")
print("   Then continue from Cell 2 onward.")

✅ All packages installed successfully.

⚠️  Next step: Runtime → Restart runtime
   Then continue from Cell 2 onward.


---
## 🔑 Section 2 — Configuration & API Key Setup

Set your model provider and API key here.
- `openai` — Uses GPT-4o (recommended)
- `anthropic` — Uses Claude 3.5 Sonnet

In [2]:
# ============================================================
# CELL 2: Configuration
# ============================================================
import os
import getpass

MODEL_NAME     = "gemini-2.0-flash-lite"   # <── change model here
TEMPERATURE    = 0.2
SHOW_RAW_JSON  = False
EXPORT_RESULTS = False

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass(
        "🔑 Enter your Google AI Studio API key: "
    )

print(f"✅ Google API key set. Model: {MODEL_NAME}")
print("   Get a free key at: https://aistudio.google.com/app/apikey")

🔑 Enter your Google AI Studio API key: ··········
✅ Google API key set. Model: gemini-2.0-flash-lite
   Get a free key at: https://aistudio.google.com/app/apikey


---
## 🏗️ Section 3 — Agent Core: Data Models & LangChain Chain

Defines Pydantic schemas, the evaluation prompt template, and the LangChain chain.

In [3]:
# ============================================================
# CELL 3A: Pydantic Data Models
# ============================================================
from pydantic import BaseModel, Field
from typing import List


class CriterionScore(BaseModel):
    """Score and explanation for a single evaluation criterion."""
    score: float = Field(ge=0, le=10, description="Score 0.0–10.0")
    explanation: str = Field(description="1-2 sentences explaining the score")


class PromptEvaluation(BaseModel):
    """
    Full evaluation result for a single prompt.
    final_score = arithmetic mean of all 5 criterion scores.
    """
    clarity:             CriterionScore
    specificity:         CriterionScore
    context:             CriterionScore
    output_format:       CriterionScore
    persona:             CriterionScore
    final_score:         float = Field(ge=0, le=10)
    overall_explanation: str
    suggestions:         List[str] = Field(min_length=2, max_length=3)


print("\u2705 Pydantic models defined.")

✅ Pydantic models defined.


In [4]:
SYSTEM_MESSAGE = """\
You are an expert AI Prompt Quality Evaluator with deep knowledge of prompt engineering.
Evaluate the submitted prompt on exactly these 5 criteria (each scored 0.0–10.0):

1. CLARITY (0-10)       — Is the goal clear and unambiguous?
2. SPECIFICITY (0-10)   — Are sufficient details and requirements provided?
3. CONTEXT (0-10)       — Is background, audience, or use case mentioned?
4. OUTPUT FORMAT (0-10) — Are format, tone, or length constraints specified?
5. PERSONA (0-10)       — Is a specific AI role assigned?

SCORING GUIDE:
  0-2: Completely missing | 3-4: Very weak | 5-6: Partial | 7-8: Good | 9-10: Excellent

FINAL SCORE = arithmetic mean of all 5 scores, rounded to 1 decimal.

Respond ONLY with raw valid JSON (no markdown, no preamble):
{{
  "clarity":             {{"score": <0-10>, "explanation": "<1-2 sentences>"}},
  "specificity":         {{"score": <0-10>, "explanation": "<1-2 sentences>"}},
  "context":             {{"score": <0-10>, "explanation": "<1-2 sentences>"}},
  "output_format":       {{"score": <0-10>, "explanation": "<1-2 sentences>"}},
  "persona":             {{"score": <0-10>, "explanation": "<1-2 sentences>"}},
  "final_score":         <float 0-10>,
  "overall_explanation": "<2-3 sentences>",
  "suggestions":         ["<suggestion 1>", "<suggestion 2>", "<suggestion 3>"]
}}
"""

from langchain_core.prompts import ChatPromptTemplate

EVAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_MESSAGE),
    ("human",  "Please evaluate this prompt:\\n\\n```\\n{user_prompt}\\n```"),
])

print("\u2705 Prompt template defined.")

✅ Prompt template defined.


In [5]:
# ============================================================
# CELL 3C: Build the LangChain Evaluation Chain
# ============================================================
# Chain: EVAL_PROMPT -> LLM -> Pydantic structured output

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    # convert_system_message_to_human=True is NOT needed for
    # gemini-1.5+ which natively supports system messages.
)

structured_llm   = llm.with_structured_output(PromptEvaluation)
evaluation_chain = EVAL_PROMPT | structured_llm

print(f"✅ Gemini chain ready.")
print(f"   Model    : {MODEL_NAME}")
print(f"   Temp     : {TEMPERATURE}")

✅ Gemini chain ready.
   Model    : gemini-2.0-flash-lite
   Temp     : 0.2


---
## 🎨 Section 4 — Display & Helper Functions

Formatting utilities that render evaluation results as clean, color-coded reports.

In [7]:
# ============================================================
# CELL 4: Display & Helper Functions
# ============================================================
from rich.console import Console
from rich.panel   import Panel
from rich.table   import Table
from rich.text    import Text
from rich.rule    import Rule
from rich         import box
from tenacity import (
    retry, stop_after_attempt, wait_exponential, retry_if_exception_type
)

try:
    from google.api_core.exceptions import ResourceExhausted as GeminiRateLimitError
except ImportError:
    GeminiRateLimitError = Exception

console = Console()


def score_bar(score: float, width: int = 18) -> str:
    """Convert 0-10 score to a visual progress bar."""
    filled = int(round(score / 10 * width))
    return '\u2588' * filled + '\u2591' * (width - filled) + f'  {score:.1f}'


def score_color(score: float) -> str:
    """Return a Rich color string based on score range."""
    if score >= 8.0: return 'bold green'
    if score >= 6.0: return 'bold yellow'
    if score >= 4.0: return 'bold orange3'
    return 'bold red'


def score_label(score: float) -> str:
    """Return a descriptive label for a score."""
    if score >= 9.0: return 'Excellent \u2728'
    if score >= 7.5: return 'Good \U0001f44d'
    if score >= 6.0: return 'Decent \U0001f7e1'
    if score >= 4.0: return 'Weak \u26a0\ufe0f'
    if score >= 2.0: return 'Poor \U0001f534'
    return 'Very Poor \u274c'


def display_evaluation(prompt: str, result: PromptEvaluation) -> None:
    """Render a full evaluation report using Rich."""
    console.print()
    console.rule('[bold cyan]\U0001f9e0 PROMPT QUALITY EVALUATION REPORT[/bold cyan]')
    console.print(Panel(f'[italic]{prompt}[/italic]',
                        title='[bold]\U0001f4dd Input Prompt[/bold]',
                        border_style='blue', padding=(0, 2)))
    fs    = result.final_score
    color = score_color(fs)
    label = score_label(fs)
    banner = Text(f'  FINAL SCORE: {fs:.1f} / 10   {label}  ', justify='center')
    banner.stylize(color)
    console.print(Panel(banner, border_style=color.split()[-1], padding=(0, 2)))

    table = Table(title='\U0001f4ca Criterion Breakdown', box=box.ROUNDED,
                  header_style='bold magenta', padding=(0, 1), expand=True)
    table.add_column('Criterion', style='bold', no_wrap=True)
    table.add_column('Score / 10', justify='center', no_wrap=True)
    table.add_column('Bar', justify='left')
    table.add_column('Explanation', justify='left', ratio=3)

    for name, crit in [
        ('\U0001f3af Clarity',             result.clarity),
        ('\U0001f50d Specificity/Details', result.specificity),
        ('\U0001f4da Context',             result.context),
        ('\U0001f4d0 Output Format',       result.output_format),
        ('\U0001f3ad Persona Defined',     result.persona),
    ]:
        c = score_color(crit.score).split()[-1]
        table.add_row(name,
                      Text(f'{crit.score:.1f}', style=f'bold {c}'),
                      Text(score_bar(crit.score), style=c),
                      crit.explanation)

    console.print(table)
    console.print(Panel(result.overall_explanation,
                        title='[bold]\U0001f4ac Overall Assessment[/bold]',
                        border_style='cyan', padding=(0, 2)))
    sug = '\n'.join(f'[bold yellow]{i}.[/bold yellow] {s}'
                    for i, s in enumerate(result.suggestions, 1))
    console.print(Panel(sug, title='[bold]\U0001f4a1 Improvement Suggestions[/bold]',
                        border_style='yellow', padding=(0, 2)))
    console.rule()


@retry(
    retry=retry_if_exception_type(GeminiRateLimitError),
    wait=wait_exponential(multiplier=1, min=4, max=64),
    stop=stop_after_attempt(5),
    before_sleep=lambda rs: console.print(
        f'[yellow]⏳ Rate limit hit. Retrying in {rs.next_action.sleep:.0f}s '
        f'(attempt {rs.attempt_number}/5)...[/yellow]'
    ),
)
def _invoke_chain(prompt: str) -> PromptEvaluation:
    return evaluation_chain.invoke({'user_prompt': prompt})


def evaluate_prompt(prompt: str, show_raw: bool = False) -> PromptEvaluation:
    """Evaluate a prompt, display results, and return the evaluation object."""
    console.print(f'\n[dim]\u23f3 Evaluating... (model: {MODEL_NAME})[/dim]')
    result: PromptEvaluation = _invoke_chain(prompt)
    if show_raw:
        console.print_json(result.model_dump_json(indent=2))
    display_evaluation(prompt, result)
    return result


print('\u2705 Display functions defined.')

✅ Display functions defined.


---
## 🎮 Section 5 — Interactive Demo

Enter any prompt below and see it evaluated in real time.

In [13]:
# ============================================================
# CELL 5A: Interactive Single-Prompt Evaluation
# ============================================================
user_input = input('\n\U0001f4dd Enter a prompt to evaluate:\n> ').strip()

if user_input:
    result = evaluate_prompt(user_input, show_raw=SHOW_RAW_JSON)
else:
    print('\u26a0\ufe0f  No prompt entered. Please try again.')


📝 Enter a prompt to evaluate:
> write a story


⏳ Evaluating... (model: gemini-2.0-flash)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 55.074672838s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

In [ ]:
# ============================================================
# CELL 5B: Continuous Evaluation Loop
# ============================================================
# Keeps evaluating prompts until you type 'quit'.
# Useful for iteratively improving a prompt.

import time
console.print('\n[bold cyan]\U0001f504 Continuous Mode — type quit to stop[/bold cyan]\n')
session_results = []

while True:
    try:
        user_input = input('\U0001f4dd Prompt (or quit): ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if user_input.lower() in ('quit', 'exit', 'q', ''):
        console.print('\n[bold green]\U0001f44b Done.[/bold green]')
        break
    result = evaluate_prompt(user_input)
    session_results.append({'prompt': user_input, 'score': result.final_score})

if session_results:
    console.print('\n[bold]\U0001f4c8 Session Summary[/bold]')
    for i, r in enumerate(session_results, 1):
        short = r['prompt'][:60] + ('...' if len(r['prompt']) > 60 else '')
        c = score_color(r['score']).split()[-1]
        console.print(f"  {i}. [{c}]{r['score']:.1f}[/{c}]  {short}")
    avg = sum(r['score'] for r in session_results) / len(session_results)
    console.print(f'\n[bold]Session average: {avg:.1f} / 10[/bold]')

---
## 🧪 Section 6 — Batch Test Suite (10 Prompts)

10 curated test prompts ranging from very poor to excellent quality.

| Quality Level | Expected Score Range |
|---|---|
| Very Poor | 0.0 – 2.0 |
| Poor | 2.0 – 4.0 |
| Weak | 4.0 – 5.5 |
| Decent | 5.5 – 7.0 |
| Good | 7.0 – 8.5 |
| Excellent | 8.5 – 10.0 |

In [8]:
# ============================================================
# CELL 6A: Define Test Prompts
# ============================================================
# 10 prompts ordered worst → best, with expected score ranges.

TEST_PROMPTS = [
    # VERY POOR ─────────────────────────────────────────────
    {
        'label': 'VERY POOR — One-word request',
        'expected_range': (0.0, 2.0),
        'prompt': 'Write'
    },
    # POOR ───────────────────────────────────────────────────
    {
        'label': 'POOR — Vague topic, no detail or format',
        'expected_range': (2.0, 4.0),
        'prompt': 'Write me a poem'
    },
    {
        'label': 'POOR — Minimal task, nothing else specified',
        'expected_range': (2.0, 4.0),
        'prompt': 'Summarize this article'
    },
    # WEAK ───────────────────────────────────────────────────
    {
        'label': 'WEAK — Clear task but no context/format/persona',
        'expected_range': (4.0, 5.5),
        'prompt': 'Write a Python function to sort a list of numbers in descending order'
    },
    {
        'label': 'WEAK — Some detail but missing persona and format',
        'expected_range': (4.0, 5.5),
        'prompt': 'Explain how transformers work in machine learning. Make it easy to understand.'
    },
    # DECENT ─────────────────────────────────────────────────
    {
        'label': 'DECENT — Audience and length, but no persona or format',
        'expected_range': (5.5, 7.0),
        'prompt': ('As a senior developer, explain REST APIs for junior developers. '
                   'Keep it under 300 words and use plain English.')
    },
    # GOOD ───────────────────────────────────────────────────
    {
        'label': 'GOOD — Persona, format, audience, and constraints',
        'expected_range': (7.0, 8.5),
        'prompt': ('Act as a data analyst. I will provide a CSV of monthly sales data. '
                   'Summarize the top 3 trends, flag anomalies, and return results as a '
                   'markdown table with a bullet-point summary below it.')
    },
    {
        'label': 'GOOD — Strong persona, scope, and structured output',
        'expected_range': (7.0, 8.5),
        'prompt': ('You are a senior Python engineer. Write a type-annotated function that '
                   'validates US email addresses using regex. Include a Google-style docstring '
                   'and 3 pytest unit tests. Return only the code, no explanation.')
    },
    # EXCELLENT ──────────────────────────────────────────────
    {
        'label': 'EXCELLENT — Full persona, context, format, tone, length',
        'expected_range': (8.5, 10.0),
        'prompt': ('You are an expert technical writer creating developer docs for a fintech startup. '
                   'Write a 500-word tutorial on async/await in Python for mid-level developers. '
                   'Structure: (1) brief intro, (2) three concepts each with a code example, '
                   '(3) one common pitfall. Tone: friendly but precise. Format: markdown with headers.')
    },
    {
        'label': 'EXCELLENT — Enterprise context, multi-constraint, professional output',
        'expected_range': (8.5, 10.0),
        'prompt': ('You are a senior AI governance consultant specializing in model risk management '
                   'for regulated financial institutions. Our bank is preparing for an SR 11-7 review '
                   'of a credit scoring model (XGBoost, FICO data). Write a Model Risk Assessment '
                   '(max 400 words) covering: (1) key validation findings, (2) conceptual soundness, '
                   '(3) ongoing monitoring recommendations. '
                   'Audience: CRO and internal audit. Format: professional memo with headers.')
    },
]

print(f'\u2705 {len(TEST_PROMPTS)} test prompts loaded.')
for i, tp in enumerate(TEST_PROMPTS, 1):
    lo, hi = tp['expected_range']
    print(f'  {i:2}. [{lo:.1f}\u2013{hi:.1f}] {tp["label"]}')

✅ 10 test prompts loaded.
   1. [0.0–2.0] VERY POOR — One-word request
   2. [2.0–4.0] POOR — Vague topic, no detail or format
   3. [2.0–4.0] POOR — Minimal task, nothing else specified
   4. [4.0–5.5] WEAK — Clear task but no context/format/persona
   5. [4.0–5.5] WEAK — Some detail but missing persona and format
   6. [5.5–7.0] DECENT — Audience and length, but no persona or format
   7. [7.0–8.5] GOOD — Persona, format, audience, and constraints
   8. [7.0–8.5] GOOD — Strong persona, scope, and structured output
   9. [8.5–10.0] EXCELLENT — Full persona, context, format, tone, length
  10. [8.5–10.0] EXCELLENT — Enterprise context, multi-constraint, professional output


In [ ]:
# ============================================================
# CELL 6B: Run Batch Test Suite
# ============================================================
# Runs all 10 prompts sequentially.
# Cost estimate: ~$0.02-$0.05 with GPT-4o | Time: ~60s
# To run a subset: change TEST_PROMPTS to TEST_PROMPTS[0:3]

import time, json
batch_results = []
console.rule('[bold cyan]\U0001f9ea BATCH TEST SUITE[/bold cyan]')
console.print(f'Running [bold]{len(TEST_PROMPTS)}[/bold] test prompts...\n')

for i, test in enumerate(TEST_PROMPTS, 1):
    console.print(f'[bold dim]\n\u2500\u2500 Test {i}/{len(TEST_PROMPTS)}: {test["label"]} \u2500\u2500[/bold dim]')
    try:
        t0 = time.time()
        r  = evaluate_prompt(test['prompt'], show_raw=False)
        lo, hi = test['expected_range']
        batch_results.append({
            'num': i, 'label': test['label'],
            'score': r.final_score,
            'expected': f'{lo:.1f}\u2013{hi:.1f}',
            'in_range': lo <= r.final_score <= hi,
            'clarity': r.clarity.score,
            'specificity': r.specificity.score,
            'context': r.context.score,
            'output_format': r.output_format.score,
            'persona': r.persona.score,
            'elapsed': round(time.time() - t0, 1),
        })
    except Exception as e:
        console.print(f'[bold red]\u274c Error on test {i}: {e}[/bold red]')
        batch_results.append({'num': i, 'label': test['label'], 'score': None, 'error': str(e)})

console.print('\n[bold green]\u2705 Batch evaluation complete![/bold green]')

─────────────────────────────────────────────── 🧪 BATCH TEST SUITE ───────────────────────────────────────────────

Running 10 test prompts...

── Test 1/10: VERY POOR — One-word request ──

⏳ Evaluating... (model: gemini-2.0-flash-lite)

❌ Error on test 1: Error calling model 'gemini-2.0-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. 
{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. 
For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your 
current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: 
generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: 
gemini-2.0-flash-lite\n* Quota exceeded for metric: 
generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* 
Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: 
gemini-2.0-flash-lite\nPlease retry in 21.984073604s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 
'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 
'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 
'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_input_token_count', 
'quotaId': 'GenerateContentInputTokensPerModelPerMinute-FreeTier', 'quotaDimensions': {'location': 'global', 
'model': 'gemini-2.0-flash-lite'}}, {'quotaMetric': 
'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 
'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 
'gemini-2.0-flash-lite'}}, {'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests',
'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 
'gemini-2.0-flash-lite'}}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '21s'}]}}

── Test 2/10: POOR — Vague topic, no detail or format ──

⏳ Evaluating... (model: gemini-2.0-flash-lite)

❌ Error on test 2: Error calling model 'gemini-2.0-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. 
{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. 
For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your 
current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: 
generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: 
gemini-2.0-flash-lite\n* Quota exceeded for metric: 
generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* 
Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: 
gemini-2.0-flash-lite\nPlease retry in 48.365824399s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 
'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 
'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 
'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_input_token_count', 
'quotaId': 'GenerateContentInputTokensPerModelPerMinute-FreeTier', 'quotaDimensions': {'location': 'global', 
'model': 'gemini-2.0-flash-lite'}}, {'quotaMetric': 
'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 
'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 
'gemini-2.0-flash-lite'}}, {'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests',
'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 
'gemini-2.0-flash-lite'}}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '48s'}]}}

── Test 3/10: POOR — Minimal task, nothing else specified ──

⏳ Evaluating... (model: gemini-2.0-flash-lite)

In [21]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [ ]:
# ============================================================
# CELL 6C: Batch Results Summary Table
# ============================================================
console.rule('[bold cyan]\U0001f4ca RESULTS SUMMARY[/bold cyan]')

t = Table(box=box.ROUNDED, header_style='bold magenta', expand=True,
          title='Prompt Quality Test Results')
for col, kw in [('#', {'width':3,'justify':'center'}),
                ('Tier', {'width':12}),
                ('Score', {'width':6,'justify':'center'}),
                ('Expected', {'width':9,'justify':'center'}),
                ('Clr', {'width':5,'justify':'center'}),
                ('Spc', {'width':5,'justify':'center'}),
                ('Ctx', {'width':5,'justify':'center'}),
                ('Fmt', {'width':5,'justify':'center'}),
                ('Per', {'width':5,'justify':'center'}),
                ('\u2713', {'width':5,'justify':'center'})]:
    t.add_column(col, **kw)

valid = [r for r in batch_results if r.get('score') is not None]

for r in batch_results:
    if r.get('score') is None:
        t.add_row(str(r['num']), r['label'][:12], 'ERR', '-','-','-','-','-','-', '\u274c')
        continue
    sc = r['score']
    c  = score_color(sc).split()[-1]
    def fs(v): return Text(f'{v:.0f}', style=f'bold {score_color(v).split()[-1]}')
    tier = r['label'].split('\u2014')[0].strip() if '\u2014' in r['label'] else r['label'].split('\u2013')[0].strip()
    t.add_row(str(r['num']), tier[:12],
              Text(f'{sc:.1f}', style=f'bold {c}'), r['expected'],
              fs(r['clarity']), fs(r['specificity']), fs(r['context']),
              fs(r['output_format']), fs(r['persona']),
              '\u2705' if r['in_range'] else '\u26a0\ufe0f')

console.print(t)

if valid:
    scores = [r['score'] for r in valid]
    console.print(f'\n[bold]Tests: {len(batch_results)} | '
                  f'In range: {sum(r["in_range"] for r in valid)}/{len(valid)} | '
                  f'Min: {min(scores):.1f} | Max: {max(scores):.1f} | '
                  f'Avg: {sum(scores)/len(scores):.1f}[/bold]')
    console.print('[dim]Columns: Clr=Clarity Spc=Specificity Ctx=Context Fmt=OutputFormat Per=Persona[/dim]')

In [ ]:
# ============================================================
# CELL 6D: Export Results to JSON (Optional)
# ============================================================
# To download: Files panel (left sidebar) → right-click → Download

output_path = 'prompt_test_results.json'
export_data = {
    'meta': {'model': MODEL_NAME, 'provider': MODEL_PROVIDER, 'temperature': TEMPERATURE},
    'results': batch_results
}
with open(output_path, 'w') as f:
    json.dump(export_data, f, indent=2, default=str)
console.print(f'\n[bold green]\U0001f4be Results exported to: {output_path}[/bold green]')
console.print('[dim]In Colab: Files panel (left sidebar) \u2192 right-click \u2192 Download[/dim]')

---
## 📚 Reference: High-Scoring Prompt Template

```
You are a [ROLE/PERSONA].

[CONTEXT: Who is this for? What is the situation?]

[TASK: What exactly do you need?]

Requirements:
- [Specific detail 1]
- [Specific detail 2]
- [Specific detail 3]

Output format: [Format type — markdown, JSON, prose, etc.]
Length: [Word/sentence count]
Tone: [Professional / Friendly / Technical]
Constraints: [What to avoid or include]
```

### 🔢 Score Interpretation

| Score | Label | Action |
|---|---|---|
| 9.0 – 10.0 | Excellent ✨ | Production-ready |
| 7.5 – 8.9 | Good 👍 | Minor polish needed |
| 6.0 – 7.4 | Decent 🟡 | Add persona and format specs |
| 4.0 – 5.9 | Weak ⚠️ | Significant rewrite needed |
| 2.0 – 3.9 | Poor 🔴 | Nearly unusable |
| 0.0 – 1.9 | Very Poor ❌ | Single-word or empty prompt |

---
*Built with LangChain | GPT-4o / Claude 3.5 | MIT License*